# Test `Fast_VTON_full.pt` trên Kaggle GPU P100

Notebook test model virtual try-on `Fast_VTON_full.pt` (Stage 1) đã huấn luyện xong, thiết kế **chỉ cho Kaggle Notebook với GPU Tesla P100**.

**Luồng:** kiểm tra Kaggle P100 → cài dependencies (Fast-VTON + dự án test) → tải model vào `models/` bằng `gdown` → kiểm tra nhanh pipeline → chạy Gradio public link để up ảnh người + ảnh quần áo.

> **Quan trọng:** trong Kaggle, chọn **Accelerator → GPU P100** và bật **Internet** trong Settings. Internet cần cho `gdown`, lần tải đầu tiên của Segformer/scheduler, và Gradio tạo public link.
>
> **P100 lưu ý:** P100 không có bf16, nhưng bundle là **fp16** nên chạy được. Phải `pip uninstall -y peft` (xung đột với diffusers 0.22).

In [ ]:
import os
import torch

IS_KAGGLE = os.path.exists('/kaggle/input') or ('KAGGLE_CONTAINER_NAME' in os.environ)
if not IS_KAGGLE:
    raise RuntimeError('Notebook này chỉ hỗ trợ Kaggle Notebook với GPU Tesla P100.')
if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU. Vào Kaggle Settings → Accelerator và chọn GPU P100.')
GPU_NAME = torch.cuda.get_device_name(0)
if 'P100' not in GPU_NAME.upper():
    raise RuntimeError(f'Yêu cầu GPU Tesla P100, nhưng runtime hiện tại là: {GPU_NAME}')
print('Kaggle GPU đã xác nhận:', GPU_NAME)

WORKDIR = os.getcwd()
PARENT = os.path.dirname(WORKDIR)
FAST_VTON_DIR = os.path.join(PARENT, 'Fast-VTON')
print('WORKDIR   :', WORKDIR)
print('Fast-VTON :', FAST_VTON_DIR)

if not os.path.isdir(FAST_VTON_DIR):
    !git clone -q https://github.com/hoangtung386/Fast-VTON.git {FAST_VTON_DIR}
else:
    print('Fast-VTON đã có sẵn')

## Bước 1 — Cài đặt môi trường

Pin đúng version theo Fast-VTON (`diffusers==0.22.0`, `transformers==4.37.2`, `torch==2.2.1`), cài Fast-VTON (package `swiftedit`, import name `src`), rồi cài dự án test (`fast_vton`). Cuối cùng **gỡ `peft`** để diffusers không bật PEFT backend gây vỡ giữa lượt UNet.

In [ ]:
!pip install -q torch==2.2.1 torchvision==0.17.1
!pip install -q -e {FAST_VTON_DIR}[vton]
!pip install -q numpy==1.26.4
!pip uninstall -q -y peft
!pip install -q -e {WORKDIR}

## Bước 2 — Tải model `Fast_VTON_full.pt` từ Google Drive

Tải trực tiếp file public từ Google Drive bằng `gdown`. File sẽ được đặt vào `models/Fast_VTON_full.pt` (vị trí mặc định `Config().bundle_path`).

In [ ]:
import os

MODEL_DIR = os.path.join(WORKDIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, 'Fast_VTON_full.pt')

GDRIVE_MODEL_URL = 'https://drive.google.com/file/d/14CX8n1co5riWvX2KHVFI6Iud7Vi8MmJJ/view?usp=sharing'

if not os.path.exists(MODEL_PATH):
    !pip install -q gdown
    !gdown --fuzzy {GDRIVE_MODEL_URL} -O {MODEL_PATH}
    if not os.path.isfile(MODEL_PATH) or os.path.getsize(MODEL_PATH) == 0:
        raise RuntimeError('Không tải được checkpoint. Kiểm tra Kaggle Internet và quyền public của link Google Drive.')
    print('đã chuẩn bị', MODEL_PATH)
else:
    print('model đã có sẵn tại', MODEL_PATH)

## Bước 3 — Kiểm tra nhanh pipeline (bắt lỗi môi trường sớm)

Chạy thử trên ảnh synthetic để xác nhận: bundle load được, DINOv2/CLIP/VAE/inversion hoạt động, mask + forward chạy qua. Ảnh rác nên kết quả chỉ để verify không crash.

In [ ]:
import os
import torch
from PIL import Image
from fast_vton.pipeline import FastVTONPipeline

bundle = os.path.join(WORKDIR, 'models', 'Fast_VTON_full.pt')
pred = FastVTONPipeline(bundle_path=bundle, device='cuda')
person = Image.new('RGB', (384, 512), (220, 200, 180))
garment = Image.new('RGB', (224, 224), (30, 90, 200))
agnostic = pred.build_agnostic(person)
out = pred.try_on(person, agnostic, garment)
print('bundle step :', pred.bundle.manifest.step)
print('output shape:', tuple(out.shape))

## Bước 4 — Chạy Gradio để test thật

Mở public link `gradio.live` hiện ra, up **ảnh người mẫu** + **ảnh quần áo**, bấm *Thử đồ*. Ảnh agnostic tự sinh (human parsing). Để tắt auto-agnostic, bỏ tick và up sẵn ảnh agnostic.

In [ ]:
%cd {WORKDIR}
from fast_vton.app import build_demo
build_demo().launch(share=True, debug=False)

## (Tuỳ chọn) Chạy qua CLI thay vì Gradio

Đặt ảnh test vào `data/` rồi chạy. Hữu ích khi chạy batch hoặc không cần UI.

In [ ]:
print('CLI test:')
print('  python -m fast_vton.cli --person data/nguoi.jpg --garment data/ao.jpg --output outputs/result.png')
print('Tắt auto-agnostic nếu có sẵn ảnh agnostic:')
print('  python -m fast_vton.cli --person data/nguoi.jpg --garment data/ao.jpg --agnostic data/agnostic.jpg --no-auto-agnostic --output outputs/result.png')